# Lab 4: LLMs and Prompt Engineering for Decision Support

**Duration:** 2 weeks [30 Jul - 13 Aug, 2026]
**Due Date:** 13th August, 2026
**Format:** Jupyter Notebook / Google Colab + external APIs + GitHub version control
**Grading:** This is a graded lab.

**Student Name:** [Enter Name]
**Student ID:** [Enter ID]

---

### Objective

In the previous labs you *trained* models. In this lab you will *use* a model that someone
else spent millions of dollars training — a **Large Language Model (LLM)** — and learn that
getting good results out of one is an engineering discipline of its own: **prompt
engineering**.

You will build a **decision support system for a microfinance loan officer**. Given a pile of
free-text loan application letters, your system will:

1. **Summarize** each application into a short, factual brief,
2. **Extract** specific structured data points (JSON) that a downstream system could store,
3. Produce a **decision-support recommendation** — while keeping the human firmly in the loop.

Just as importantly, you will **evaluate** the LLM's output for quality, reliability, and
appropriateness: Does it hallucinate? Is it consistent across runs? Should it be trusted to
make the final call?

---

### Choosing an API provider

You need an LLM API with a **free tier**. Recommended options (pick ONE):

| Provider | Free tier | Notes |
|---|---|---|
| **Groq** (recommended) | Yes, generous | OpenAI-compatible API, very fast, open models (Llama) |
| **Google Gemini** | Yes | `google-generativeai` package |
| **Hugging Face Inference API** | Yes, limited | Many open models |
| OpenAI / Anthropic | Paid | Fine if you already have credits |

The notebook's example code uses the **OpenAI-compatible chat format** (works with Groq and
OpenAI directly; Gemini users adapt the call in one place). Everything else in the lab is
provider-agnostic.

---
### Part 0: Repository and API-key setup

1. Create a **public** repository named `lab-4-llm-decision-support` and save this notebook
   inside it.
2. Sign up with your chosen provider and create an **API key**.
3. **NEVER hard-code or commit your API key.** This is a graded requirement.
   - Locally: put it in a `.env` file and add `.env` to `.gitignore`.
   - Colab: use the Secrets panel (key icon) and read it with `google.colab.userdata`.
4. Add a `requirements.txt`: `openai python-dotenv pandas matplotlib`.
5. Commit and push after **each Part** — we will check for incremental commits.

> **A leaked key in your commit history = resubmission + penalty.** Keys can be scraped from
> public repos within minutes.

In [9]:
import os
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()

API_KEY = os.environ["GROQ_API_KEY"]

client = OpenAI(
    api_key=API_KEY,
    base_url="https://api.groq.com/openai/v1",
)

MODEL = "llama-3.3-70b-versatile"

print("Client ready.")

Client ready.


---
# Section 1 — Talking to an LLM Programmatically

Before building anything, understand the anatomy of an API call: **messages and roles**
(`system`, `user`, `assistant`), and the **generation parameters** (`temperature`,
`max_tokens`).

### Part 1.1 — Your first API call

In [11]:
# TODO: Write a helper function you will reuse for the WHOLE lab:
def ask_llm(user_prompt,
            system_prompt="You are a helpful assistant.",
            temperature=0.7,
            max_tokens=500):
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ],
        temperature=temperature,
        max_tokens=max_tokens,
    )
    return response

# TODO: Call it once with a simple question and print the answer.
response = ask_llm("What is the capital of Ghana?")
print(response.choices[0].message.content)

# TODO: Print response.usage as well — how many tokens did your call consume?

print(response.usage)

The capital of Ghana is Accra.
CompletionUsage(completion_tokens=9, prompt_tokens=48, total_tokens=57, completion_tokens_details=None, prompt_tokens_details=None, queue_time=0.040941021, prompt_time=0.001402465, completion_time=0.012388187, total_time=0.013790652)


**Student Reasoning — Anatomy of a call**
*1. What is the difference between the `system` and `user` roles? Give an example of
something that belongs in each.*
System role: Gives the AI its overall rules, behavior, and instructions.
Example: “You are a helpful AI assistant. Always be clear and accurate.”
User role: Contains the actual request or question from the user.
Example: “What is the capital of Ghana?”
*2. What is a token, roughly? Why do API providers bill per token rather than per request?*
A token is a small piece of text that an AI model processes. It can be a whole word, part of a word, or punctuation.
API providers charge per token because the amount of text the model processes can vary greatly between requests. A short request uses fewer tokens than a long conversation, so token-based pricing is more precise and fair.

> **Answer:** [Double-click to edit]

### Part 1.2 — Temperature: the randomness dial

In [3]:
# TODO: Ask the SAME question 5 times at temperature=0.0 and 5 times at temperature=1.2.
question = "Suggest a name for a savings product for market traders in Accra."
responses_temp_0 = []
responses_temp_12 = []
for _ in range(5):
    response = ask_llm(question, temperature=0.0)
    responses_temp_0.append(response.choices[0].message.content)
for _ in range(5):
    response = ask_llm(question, temperature=1.2)
    responses_temp_12.append(response.choices[0].message.content)
# TODO: Print all 10 answers, grouped by temperature.
print("===== Temperature = 0.0 =====")
for i, answer in enumerate(responses_temp_0, start=1):
    print(f"{i}. {answer}\n")
print("\n===== Temperature = 1.2 =====")
for i, answer in enumerate(responses_temp_12, start=1):
    print(f"{i}. {answer}\n")

===== Temperature = 0.0 =====
1. Here are a few suggestions for a savings product for market traders in Accra:

1. **Makola Save**: "Makola" is a well-known market in Accra, so this name could resonate with market traders.
2. **Trader's Treasure**: This name emphasizes the idea of saving and accumulating wealth.
3. **Accra Amanfu**: "Amanfu" is a Ghanaian word for "savings" or "treasury", so this name incorporates local language and culture.
4. **Market Mobi**: This name is short and catchy, and "Mobi" implies mobility and flexibility, which could appeal to market traders.
5. **Sika Kurom**: "Sika" means "money" in Ghanaian, and "Kurom" means "box" or "container", so this name suggests a safe and secure place to store savings.
6. **Traders' Fund**: This name is straightforward and emphasizes the idea of a collective fund for market traders.
7. **Adanfo Save**: "Adanfo" means "friends" or "partners" in Ghanaian, so this name suggests a sense of community and cooperation.

Choose the one

**Student Reasoning — Temperature**
*What did you observe at each temperature? For the loan decision-support system you are about
to build, which temperature regime is appropriate, and why?*

> **Answer:** At temperature 0.0, the responses were more deterministic and consistent, with similar answers being repeated. For a loan decision-support system, a low temperature (around 0.0–0.2) is appropriate because we want consistent, predictable, and reliable decisions, rather than creative or random responses.

---
# Section 2 — The Dataset: Loan Application Letters

Run the next cell to load **six loan application letters** submitted to a (fictional)
microfinance institution in Ghana, plus **gold-standard extraction labels** for three of them
(you will use these for evaluation in Section 4).

Read at least two letters fully before moving on — you cannot engineer prompts for text you
have not read.

In [12]:
LETTERS = {
"L001": """Dear Sir/Madam,
My name is Akosua Mensah and I have been selling provisions at Makola Market for 12 years.
I am applying for a loan of GHS 8,000 to buy a deep freezer and expand into frozen foods.
My current stall makes about GHS 900 profit each month. I have saved GHS 2,500 with your
susu scheme over the past two years and I have never missed a contribution. I can repay
GHS 450 monthly over 20 months. My sister, a teacher, will stand as my guarantor.
Thank you for considering my application.""",

"L002": """Hello,
I am Kwame Boateng, a commercial driver in Kumasi. I need GHS 25,000 urgently to repair my
trotro engine and settle some personal debts. Business has been slow but it will surely
pick up after the festive season. I can pay back whenever the money comes. I do not have
collateral at the moment but God willing everything will be fine. Please help me quickly.""",

"L003": """Dear Loan Committee,
I am Efua Darko, owner of Darko Fashions, a registered dressmaking business in Takoradi
(registration no. BN-2019-4482). I employ three apprentices. I request GHS 15,000 to
purchase two industrial sewing machines and fabric stock ahead of the Christmas season.
Last year my December revenue alone was GHS 22,000; monthly profit averages GHS 2,800.
I hold a fixed deposit of GHS 5,000 with GCB which I can pledge. Proposed repayment:
GHS 1,100 monthly for 15 months. Attached are my sales records for the past 18 months.""",

"L004": """Good day,
My name is Yaw Owusu. I want a loan for my poultry farm at Nsawam. The amount is GHS 12,000
for feed and 500 new layers. I started the farm last year. Sometimes I make good money,
around GHS 1,500 in a good month, but bird flu affected us in March and I lost many birds.
I am rebuilding now. I can repay in 18 months. My uncle has agreed to guarantee the loan
with his taxi.""",

"L005": """Dear Manager,
I am writing on behalf of the Adenta Women's Weaving Cooperative (14 members). We seek
GHS 30,000 to buy a bulk order of yarn directly from the factory, cutting out middlemen and
raising our margins from 15% to about 35%. The cooperative has operated for 6 years and
holds GHS 9,000 in our group account. We propose repayment of GHS 2,000 monthly over
16 months, backed by our group savings and joint liability agreement.""",

"L006": """Hi,
This is Kofi. I saw your advert. I want GHS 50,000 to start a car washing business, a
provision shop, and also import phones from Dubai. I am 22 and full of energy. I have not
started any of these yet but my friends say I am very business minded. I will pay back in
one year when the businesses are booming. No collateral but I am trustworthy.""",
}
# Gold-standard labels for three letters (for Section 4 evaluation):
GOLD = {
  "L001": {"applicant_name": "Akosua Mensah", "amount_ghs": 8000,  "purpose": "buy deep freezer / expand into frozen foods",
           "monthly_profit_ghs": 900,  "has_collateral_or_guarantor": True,  "repayment_months": 20},
  "L003": {"applicant_name": "Efua Darko",    "amount_ghs": 15000, "purpose": "industrial sewing machines and fabric stock",
           "monthly_profit_ghs": 2800, "has_collateral_or_guarantor": True,  "repayment_months": 15},
  "L006": {"applicant_name": "Kofi",          "amount_ghs": 50000, "purpose": "car wash, provision shop, phone imports",
           "monthly_profit_ghs": None, "has_collateral_or_guarantor": False, "repayment_months": 12},
}
print(f"{len(LETTERS)} letters loaded.")

6 letters loaded.


---
# Section 3 — Prompt Engineering for the Decision Support System

You will now build the three components of the system, iterating on your prompts as you go.
**Keep every major prompt version** — Section 3.4 asks you to commit your prompt templates
and document how they evolved.

### Part 3.1 — Component 1: Summarization
Turn a rambling letter into a 3-4 sentence factual brief a busy loan officer can scan.

In [6]:
# TODO: Write SUMMARY_PROMPT_V1 — your first, naive attempt (e.g. just "Summarize this:").
SUMMARY_PROMPT_V1 = "Summarize this loan application:\n\n{letter_text}"
def summarize_letter(letter_id):
    letter_text = LETTERS[letter_id]
    user_prompt = SUMMARY_PROMPT_V1.format(letter_text=letter_text)
    response = ask_llm(user_prompt, temperature=0.0)
    return response.choices[0].message.content
v1_l002_summary = summarize_letter("L002")
v1_l006_summary = summarize_letter("L006")
print("v1_L002_summary:", v1_l002_summary)
print("v1_L006_summary:", v1_l006_summary)

# TODO: Now write SUMMARY_PROMPT_V2 as a proper template with:
summary_prompt_v2_system = (
    "You are an assistant to a microfinance loan officer. "
    "Your task is to summarize loan applications factually and neutrally, "
    "without inventing details. Provide a concise summary in 3-4 sentences."
)
summary_prompt_v2_user = "Summarize this loan application:\n\n{letter_text}"
def summarize_letter_v2(letter_id):
    letter_text = LETTERS[letter_id]
    user_prompt = summary_prompt_v2_user.format(letter_text=letter_text)
    response = ask_llm(user_prompt, system_prompt=summary_prompt_v2_system, temperature=0.0)
    return response.choices[0].message.content

v2_l002_summary = summarize_letter_v2("L002")
v2_l006_summary = summarize_letter_v2("L006")
print("v2_L002_summary:", v2_l002_summary)
print("v2_L006_summary:", v2_l006_summary)

# TODO: Compare V1 vs V2 outputs side by side. Keep both prompt versions in this notebook.
print("=" *80)
print("V1 vs V2 Summary Comparison")
print("=" *80)
print("V1 L002 Summary:", v1_l002_summary)
print("V2 L002 Summary:", v2_l002_summary)
print("V1 L006 Summary:", v1_l006_summary)
print("V2 L006 Summary:", v2_l006_summary)

v1_L002_summary: Kwame Boateng, a commercial driver in Kumasi, is applying for a loan of GHS 25,000. He needs the funds to repair his vehicle's engine and pay off personal debts. His business has been slow, but he expects it to improve after the festive season. He doesn't have collateral to offer and is relying on his future earnings to repay the loan. He is seeking urgent assistance.
v1_L006_summary: Here's a summary of Kofi's loan application:

* Amount requested: GHS 50,000
* Proposed businesses: car washing, provision shop, and importing phones from Dubai
* Applicant's age: 22
* Repayment plan: 1 year, relying on the expected success of the businesses
* Collateral: None, but Kofi claims to be trustworthy
* Experience: No prior experience in any of the proposed businesses, but friends consider him "business-minded"
v2_L002_summary: Kwame Boateng, a commercial driver from Kumasi, has applied for a loan of GHS 25,000. He intends to use the funds to repair his trotro engine and settle 

**Student Reasoning — Summarization prompts**
*1. What concrete problems did V1's output have that V2 fixed? Quote examples.*
V1 was more verbose and less consistent. For example, V1 L006 used a bullet-list format and added wording such as “friends consider him ‘business-minded’”, while V2 presented the information more clearly in a concise paragraph. V1 L002 also added “He is seeking urgent assistance,” which was not necessary in the summary. V2 removed these unnecessary additions and kept the key facts.
*2. Why is "no invented details" an essential instruction in this application? What is this
failure mode called in the LLM literature?*
“No invented details” is essential because adding information that was not in the loan application could lead to incorrect or unfair lending decisions. This failure mode is called hallucination in LLM literature, where a model generates information that is not supported by the source.

### Part 3.2 — Component 2: Structured extraction (JSON)
Downstream software cannot read prose. Extract the fields in `GOLD` as strict JSON.

In [ ]:
import json
import pandas as pd
# TODO: Write EXTRACT_PROMPT — a template that instructs the model to return ONLY a JSON
EXTRACT_PROMPT = """
You are an information extraction assistant.
Extract the following fields from the letter and return ONLY a valid JSON object.
The JSON must contain EXACTLY these keys:
{
    "applicant_name": "string",
    "amount_ghs": number,
    "purpose": "string",
    "monthly_profit_ghs": number or null,
    "has_collateral_or_guarantor": boolean,
    "repayment_months": number or null
}
Rules:
- If a field is not stated in the letter, use null.
- Do not guess or infer missing information.
- Return ONLY JSON. Do not include explanations or markdown.
Worked example:
Letter:
"My name is John Mensah. I am requesting GHS 5000 to expand my
provisions shop. My monthly profit is GHS 1200. My brother will
serve as my guarantor. I will repay the loan in 12 months."
Output:
{
    "applicant_name": "John Mensah",
    "amount_ghs": 5000,
    "purpose": "expand my provisions shop",
    "monthly_profit_ghs": 1200,
    "has_collateral_or_guarantor": true,
    "repayment_months": 12
}
Now extract the information from this letter:
"""
# TODO: Write extract_fields(letter_text) that calls the LLM, strips any ```json fences,

def extract_fields(letter_text):
    try:
        response = client.chat.completions.create(
            model=MODEL,
            messages=[
                {
                    "role": "user",
                    "content": EXTRACT_PROMPT + "\n\nLETTER:\n" + letter_text
                }
            ],
            temperature=0
        )
        result = response.choices[0].message.content.strip()
        # Remove ```json fences if present
        if result.startswith("```json"):
            result = result[7:]
        if result.startswith("```"):
            result = result[3:]
        if result.endswith("```"):
            result = result[:-3]

        result = result.strip()
        # Convert JSON string to Python dictionary
        return json.loads(result)
    except Exception as e:
        print(f"Warning: Could not extract fields: {e}")
        return None
# TODO: Run it on ALL SIX letters; collect results into a pandas DataFrame
results = []
for letter_id, letter_text in LETTERS.items():
    result = extract_fields(letter_text)
    if result is not None:
        result["letter_id"] = letter_id
    results.append(result)
df = pd.DataFrame(results)
# Put letter_id first
if "letter_id" in df.columns:
    columns = ["letter_id"] + [
        col for col in df.columns if col != "letter_id"
    ]
    df = df[columns]
display(df)

,letter_id,applicant_name,amount_ghs,purpose,monthly_profit_ghs,has_collateral_or_guarantor,repayment_months
0,L001,Akosua Mensah,8000,buy a deep freezer and expand into frozen foods,900.0,True,20.0
1,L002,Kwame Boateng,25000,repair my trotro engine and settle some person...,NaN,False,NaN
2,L003,Efua Darko,15000,purchase two industrial sewing machines and fa...,2800.0,True,15.0
3,L004,Yaw Owusu,12000,for my poultry farm at Nsawam for feed and 500...,1500.0,True,18.0
4,L005,Adenta Women's Weaving Cooperative,30000,buy a bulk order of yarn directly from the fac...,NaN,True,16.0
5,L006,Kofi,50000,"start a car washing business, a provision shop...",NaN,False,12.0


**Student Reasoning — Structured extraction**
*1. Why must the few-shot example NOT come from the six letters you are processing?*
Using one of the six letters could cause the model to copy or memorize information from the data being tested, making the extraction results less reliable. The example should be separate so it only teaches the expected format.
*2. Why "use null, do not guess" — what did the model do without that instruction?*
Without this instruction, the model may invent or infer missing information instead of leaving it blank. The null rule makes the extraction more accurate and prevents unsupported information from being added.
*3. Why is temperature=0 the right choice for extraction but arguably not for creative tasks?*
temperature=0 makes the model's responses more consistent and predictable, which is useful when extracting structured data. Creative tasks benefit from more variation and originality, so a higher temperature can be more appropriate.

> **Answer:** [Double-click to edit]

### Part 3.3 — Component 3: The decision-support brief
Combine everything: for each letter, produce a recommendation brief for the loan officer —
strengths, risks, missing information, and a suggested next step. The system must
**support** the decision, not **make** it.

In [22]:
# TODO: Write BRIEF_PROMPT — it receives the letter AND your extracted JSON,
# and must output:
# 1. Strengths (bullet points, grounded in the letter)
# 2. Risks / red flags (bullet points)
# 3. Missing information the officer should request
# 4. Suggested next step — NOT "approve" or "reject"
# Give the model an explicit instruction that final decisions are made by humans.
BRIEF_PROMPT = """
You are assisting a human loan officer.
Review BOTH the original applicant letter and the extracted JSON information.
Prepare a concise decision-support brief with EXACTLY these four sections:
1. Strengths
- List strengths supported by information stated in the letter.
2. Risks / Red Flags
- List any risks, concerns, inconsistencies, or red flags.
- Do not invent information.
3. Missing Information
- List information or documents the loan officer should request before making a decision.
4. Suggested Next Step
- Recommend an appropriate next action such as:
  "invite for interview",
  "request documents",
  "request clarification",
  "flag for senior review".
- NEVER recommend "approve" or "reject".

IMPORTANT:
- The brief is decision support only.
- The final loan decision must always be made by a human loan officer.
- Base your analysis only on the letter and extracted JSON.
- If information is missing, clearly say so.
- Do not guess.
Original Letter:
{letter}
Extracted JSON:
{extracted_json}
"""
# TODO: Generate briefs for ALL SIX letters.
briefs = {}
for i, (letter_id, letter_text) in enumerate(LETTERS.items()):
    # Get the extracted information for this letter
    extracted = results[i]
    # Build the prompt using the original letter and extracted JSON
    prompt = BRIEF_PROMPT.format(
        letter=letter_text,
        extracted_json=json.dumps(extracted, indent=2)
    )
    try:
        response = client.chat.completions.create(
            model="openai/gpt-oss-120b",
            messages=[
                {
                    "role": "user",
                    "content": prompt
                }
            ],
            temperature=0
        )
        brief = response.choices[0].message.content.strip()
        # Store the brief using the actual letter ID
        briefs[letter_id] = brief
    except Exception as e:
        print(f"Warning: Could not generate brief for {letter_id}: {e}")
        briefs[letter_id] = None

# Print the briefs for L001, L002, and L006
for letter_id in ["L001", "L002", "L006"]:
    print("=" * 80)
    print(letter_id)
    print("=" * 80)
    print(briefs[letter_id])
    print()

L001
**1. Strengths**  
- **Long‑standing business experience:** 12 years selling provisions at Makola Market.  
- **Consistent cash flow:** Stated monthly profit of GHS 900.  
- **Demonstrated savings discipline:** GHS 2,500 saved through the bank’s susu scheme over the past two years with no missed contributions.  
- **Clear repayment proposal:** Ability to pay GHS 450 per month for 20 months (total repayment GHS 9,000).  
- **Guarantor available:** Sister, a teacher, has agreed to stand as guarantor.  
- **Specific use of funds:** Purchase of a deep freezer to expand into frozen foods, indicating a concrete growth plan.  

**2. Risks / Red Flags**  
- **Debt‑service ratio:** Proposed repayment (GHS 450) is ~50 % of the reported monthly profit (GHS 900), leaving limited buffer for operating expenses or unexpected shocks.  
- **Lack of documented income:** No sales records, bank statements, or tax filings provided to verify the GHS 900 profit claim.  
- **Guarantor details missing:** 

**Student Reasoning — Decision support**
*1. Compare the briefs for L003 (strong application) and L006 (weak application). Did the
system identify the right strengths and red flags in each?*
The system should identify L003 as the stronger application if its letter provides clear evidence such as a reasonable loan purpose, stated income or monthly profit, repayment ability, and collateral or a guarantor. For L006, the system should identify weaknesses such as missing financial information, unclear repayment ability, lack of collateral/guarantor, or other red flags stated in the letter. Therefore, the system should focus on evidence from each letter rather than making assumptions.
*2. Why did we forbid the model from outputting "approve"/"reject"? Give one practical and
one ethical reason.*
Practical reason: The model may not have all the information needed to make a reliable lending decision, so a human loan officer needs to review the application and request additional documents where necessary.
Ethical reason: Loan decisions can significantly affect people financially. Allowing an AI to automatically approve or reject applicants could create unfair or biased decisions and reduce human accountability.

> **Answer:** [Double

### Part 3.4 — Commit your prompt templates
Prompts ARE code. Save your final `SUMMARY_PROMPT`, `EXTRACT_PROMPT`, and `BRIEF_PROMPT` into
a separate file `prompts.py` (or `prompts.md`) in your repository and commit it with a
message describing how the prompts evolved. Paste your commit hash below.

> **Commit hash:** [paste here]

---
# Section 4 — Evaluation: Quality, Reliability, Appropriateness

An impressive demo is not a trustworthy system. Now measure it.

### Part 4.1 — Extraction accuracy against gold labels

In [25]:
# Compare extracted values against GOLD labels
# and compute per-field accuracy across L001, L003, and L006.
fields = [
    "applicant_name",
    "amount_ghs",
    "purpose",
    "monthly_profit_ghs",
    "has_collateral_or_guarantor",
    "repayment_months"
]
letter_ids = ["L001", "L003", "L006"]
accuracy_table = []
for field in fields:
    row = {"field": field}
    correct_count = 0
    for letter_id in letter_ids:
        # Get extracted value
        extracted_value = df.loc[
            df["letter_id"] == letter_id, field
        ].iloc[0]
        # Get gold value
        gold_value = GOLD[letter_id][field]
        # Compare values
        if field == "applicant_name":
            # Name matching is case-insensitive
            extracted = str(extracted_value).strip().lower()
            gold = str(gold_value).strip().lower()
            correct = extracted == gold
        elif field == "purpose":
            # Purpose is compared case-insensitively
            extracted = str(extracted_value).strip().lower()
            gold = str(gold_value).strip().lower()
            correct = extracted == gold
        else:
            # Numbers and boolean values must match exactly
            if pd.isna(gold_value):
                correct = pd.isna(extracted_value)
            else:
                correct = extracted_value == gold_value
        row[letter_id] = correct
        if correct:
            correct_count += 1
    # Accuracy across the three letters
    row["accuracy"] = correct_count / len(letter_ids)
    accuracy_table.append(row)
accuracy_df = pd.DataFrame(accuracy_table)

display(accuracy_df)

,field,L001,L003,L006,accuracy
0,applicant_name,True,True,True,1.0
1,amount_ghs,True,True,True,1.0
2,purpose,False,False,False,0.0
3,monthly_profit_ghs,True,True,True,1.0
4,has_collateral_or_guarantor,True,True,True,1.0
5,repayment_months,True,True,True,1.0


### Part 4.2 — Reliability: is the system consistent?

In [26]:
# Part 4.2 — Reliability: is the system consistent?
import json
letter_l004 = LETTERS["L004"]

def run_reliability_test(letter_text, temperature, runs=5):
    results = []
    for _ in range(runs):
        try:
            response = client.chat.completions.create(
                model=MODEL,
                messages=[
                    {
                        "role": "user",
                        "content": EXTRACT_PROMPT
                        + "\n\nLETTER:\n"
                        + letter_text
                    }
                ],
                temperature=temperature
            )
            raw_result = response.choices[0].message.content.strip()
            # Remove markdown JSON fences if present
            if raw_result.startswith("```json"):
                raw_result = raw_result[7:]
            if raw_result.startswith("```"):
                raw_result = raw_result[3:]
            if raw_result.endswith("```"):
                raw_result = raw_result[:-3]
            raw_result = raw_result.strip()
            # Check whether it is valid JSON
            parsed_result = json.loads(raw_result)
            results.append(parsed_result)
        except Exception as e:
            print(f"Warning: run failed: {e}")
            results.append(None)
    # Count valid JSON results
    valid_json_count = sum(result is not None for result in results)
    # Count unique valid outputs
    valid_results = [
        json.dumps(result, sort_keys=True)
        for result in results
        if result is not None
    ]
    unique_outputs = len(set(valid_results))
    # All five runs are identical only if there is exactly
    # one unique valid output and all five were valid.
    identical_across_runs = (
        valid_json_count == runs and unique_outputs == 1
    )
    return results, valid_json_count, unique_outputs, identical_across_runs
# Run five times at temperature = 0
results_temp_0, valid_0, unique_0, identical_0 = run_reliability_test(
    letter_l004,
    temperature=0.0
)
# Run five times at temperature = 1.0
results_temp_1, valid_1, unique_1, identical_1 = run_reliability_test(
    letter_l004,
    temperature=1.0
)
# Display the results
print("=" * 80)
print("RELIABILITY TEST — L004")
print("=" * 80)
print("\nTemperature = 0.0")
print(f"Valid JSON runs: {valid_0}/5")
print(f"Unique outputs: {unique_0}")
print(f"Identical across all 5 runs: {identical_0}")
print("\nTemperature = 1.0")
print(f"Valid JSON runs: {valid_1}/5")
print(f"Unique outputs: {unique_1}")
print(f"Identical across all 5 runs: {identical_1}")

RELIABILITY TEST — L004

Temperature = 0.0
Valid JSON runs: 5/5
Unique outputs: 1
Identical across all 5 runs: True

Temperature = 1.0
Valid JSON runs: 5/5
Unique outputs: 1
Identical across all 5 runs: True


### Part 4.3 — Hallucination probing

In [27]:
# Part 4.3 — Hallucination probing
# -----------------------------
# Test 1: Missing information
# -----------------------------
test1_prompt = """
What is the applicant's credit score?
If the credit score is not stated in the letter, clearly say that it is not provided.
Letter:
""" + LETTERS["L002"]
try:
    response1 = client.chat.completions.create(
        model=MODEL,
        messages=[
            {
                "role": "user",
                "content": test1_prompt
            }
        ],
        temperature=0
    )
    test1_output = response1.choices[0].message.content.strip()
    print("=" * 80)
    print("TEST 1 — Missing information")
    print("=" * 80)
    print(test1_output)

except Exception as e:
    test1_output = f"ERROR: {e}"
    print(test1_output)
# -----------------------------
# Test 2: Irrelevant input
# -----------------------------
irrelevant_text = """
Today's weather in Accra is sunny with temperatures around 28°C.
There may be some clouds later in the afternoon.
"""
try:
    response2 = client.chat.completions.create(
        model=MODEL,
        messages=[
            {
                "role": "user",
                "content": EXTRACT_PROMPT
                + "\n\nLETTER:\n"
                + irrelevant_text
            }
        ],
        temperature=0
    )
    test2_raw = response2.choices[0].message.content.strip()
    # Remove JSON fences if present
    if test2_raw.startswith("```json"):
        test2_raw = test2_raw[7:]
    if test2_raw.startswith("```"):
        test2_raw = test2_raw[3:]
    if test2_raw.endswith("```"):
        test2_raw = test2_raw[:-3]
    test2_raw = test2_raw.strip()
    test2_output = json.loads(test2_raw)
    print("\n" + "=" * 80)
    print("TEST 2 — Irrelevant input")
    print("=" * 80)
    print(json.dumps(test2_output, indent=2))
except Exception as e:
    test2_output = f"ERROR: {e}"
    print(test2_output)

TEST 1 — Missing information
The applicant's credit score is not provided in the letter.

TEST 2 — Irrelevant input
{
  "applicant_name": null,
  "amount_ghs": null,
  "purpose": null,
  "monthly_profit_ghs": null,
  "has_collateral_or_guarantor": null,
  "repayment_months": null
}


**Student Reasoning — Evaluation results**
*1. Report your extraction accuracy. Which field was hardest for the model and why?*
The system achieved 100% accuracy on applicant name, amount, monthly profit, collateral/guarantor, and repayment months. The purpose field was the hardest, with 0% accuracy under the exact string-matching evaluation because the extracted descriptions differed from the gold labels even when they had similar meanings.
*2. What did the reliability experiment show about temperature and production systems?*
The system produced valid JSON and identical outputs in all five runs at both temperature 0.0 and 1.0. Therefore, no variability was observed in this test, although five runs on one letter are not enough to prove the system is always deterministic.
*3. Did your system hallucinate under probing? If yes, how could the prompt (or the system
design around it) reduce the risk?*
The system did not hallucinate during either test. It correctly said the credit score was not provided and returned null values for irrelevant input. The risk could be further reduced by keeping explicit instructions such as “do not guess,” requiring missing information to be returned as null, and keeping a human loan officer responsible for the final decision

> **Answer:** [Double-click to edit]

### Part 4.4 — Appropriateness: should this system exist?
No code in this part — just judgment, which is the scarcest skill in AI for business.

**Student Reasoning — Appropriateness**
*1. Letters L002 and L006 would likely be declined. If the bank fully automated decisions
with your system, who could be unfairly harmed, and how? Consider applicants who write
poorly in English but run solid businesses.*
Applicants such as L002 and L006 could be unfairly harmed if the system made fully automated decisions. Applicants who write poorly in English or have limited documentation could appear weaker even when they operate viable businesses. This could create unfair outcomes, so a human loan officer should review the application and consider the applicant's actual circumstances

*2. Loan letters contain personal data. What are the implications of sending them to a
third-party API in another country? What would you check before deploying this at a real
Ghanaian microfinance institution?*
Sending personal loan information to a third-party API in another country creates privacy, security, and data-protection concerns. Before deployment, I would check how the provider stores and protects the data, where the data is processed, whether it is used for training, applicable Ghanaian data-protection requirements, and whether appropriate agreements and safeguards are in place

*3. Name TWO concrete safeguards you would build around this system in production (think:
human review points, logging, appeal processes, monitoring).*
Two safeguards would be: (1) mandatory human review before any lending decision, with the AI only providing decision support; and (2) audit logging and monitoring, so outputs can be reviewed for errors, bias, and unusual patterns. An appeal or reconsideration process could also allow applicants to challenge decisions.

> **Answer:** [Double-click to edit]

---
# Section 5 — Reflection

*Answer in a few sentences each:*

1. **Prompting as engineering:** How is iterating on a prompt similar to and different from
   iterating on the model hyperparameters you tuned in Lab 3?
   Prompting is similar to tuning model hyperparameters because both involve testing, evaluating results, and making changes to improve performance. The difference is that hyperparameter tuning changes how a trained model behaves internally, while prompt engineering changes the instructions and context given to the model without retraining it.

2. **Trust:** After your Section 4 evaluation, would you trust this system to run unattended?
   What single evaluation result most influenced your answer?
   I would not trust this system to run completely unattended. Although it achieved high extraction accuracy and passed the hallucination tests, the purpose field had 0% accuracy under the strict matching evaluation, and the system could still make mistakes on real applications. The evaluation result that most influenced my answer was the extraction accuracy, especially the failure on the purpose field.

3. **Cost and scale:** Estimate (from your `response.usage` numbers) the tokens needed to
   process 1,000 applications per month. What does that imply for provider choice?
   CompletionUsage(completion_tokens=73, prompt_tokens=429, total_tokens=502, completion_tokens_details=None, prompt_tokens_details=None, queue_time=0.041454579, prompt_time=0.02212717, completion_time=0.098860822, total_time=0.120987992)
   
4. **Looking back at the course:** You have now used classical ML (Lab 2), trained neural
   networks (Lab 3), and used a foundation model via API (Lab 4). For a task like this one,
   why does calling an API beat training your own model — and when would it not?
   Calling an API is better for this task because a foundation model is already trained and can understand varied loan letters without requiring a large dataset, training process, or significant computing resources. Training my own model would make more sense if I had a large, high-quality dataset, needed a specialized model, or had strict requirements around control and deployment.

> **Answer:** [Double-click to edit]

---
### Submission checklist

- [ ] All cells run top-to-bottom with no errors (`Kernel -> Restart & Run All`).
- [ ] **No API key anywhere in the notebook or the commit history.**
- [ ] Every **Student Reasoning** box is filled in with full sentences.
- [ ] `prompts.py` / `prompts.md` committed with your final prompt templates.
- [ ] Evaluation tables and adversarial test outputs visible in the saved notebook.
- [ ] Notebook pushed to `lab-4-llm-decision-support` with incremental commits.
- [ ] Repository link submitted to the course portal.
- [ ] AI Declaration form in Repository.